📌 Final Data Preparation for Dashboard

In this step, the cleaned dataset was transformed into multiple aggregated tables tailored for Tableau visualization. These datasets include KPI summaries, revenue trends, customer segmentation, and category performance. This ensures efficient dashboard performance and clear business storytelling.

In [15]:
import pandas as pd

# Load data
df = pd.read_csv("../data/processed/final_analytics_dataset.csv")
rfm = pd.read_csv("../data/processed/rfm_table.csv")

### 1️⃣ KPI SUMMARY TABLE (Executive View)

In [16]:
# KPI calculations
cancelled_revenue = df[df['order_status']=='canceled']['total_order_value'].sum()
total_revenue = df['total_order_value'].sum()
revenue_leakage = (cancelled_revenue / total_revenue) * 100

customer_orders = df.groupby('customer_unique_id')['order_id'].nunique()
repeat_rate = ((customer_orders > 1).sum() / customer_orders.count()) * 100

aov = df['total_order_value'].mean()
churn_rate = (rfm['Churn_Risk'] == 'High').mean() * 100

kpi_summary = pd.DataFrame({
    "KPI": ["Revenue Leakage %", "Repeat Rate", "Avg Order Value", "Churn Risk %"],
    "Value": [revenue_leakage, repeat_rate, aov, churn_rate]
})

kpi_summary.to_csv("../data/processed/kpi_summary.csv", index=False)

### 2️⃣ TIME SERIES DATA (Revenue Trend)

In [17]:
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M').astype(str)

monthly_revenue = df.groupby('order_month')['total_order_value'].sum().reset_index()

monthly_revenue.to_csv("../data/processed/monthly_revenue.csv", index=False)

### 3️⃣ CATEGORY PERFORMANCE TABLE

In [18]:
category_perf = df.groupby('product_category_name').agg({
    'total_order_value': 'sum',
    'order_id': 'count'
}).reset_index()

category_perf.columns = ['Category', 'Revenue', 'Order_Count']

category_perf.to_csv("../data/processed/category_performance.csv", index=False)

### 4️⃣ CUSTOMER SEGMENT TABLE (RFM)

In [19]:
rfm.to_csv("../data/processed/rfm_segments.csv", index=False)

### 5️⃣ GEOGRAPHICAL DATA

In [20]:
geo_data = df.groupby('customer_state')['total_order_value'].sum().reset_index()
geo_data.columns = ['State', 'Revenue']

geo_data.to_csv("../data/processed/state_revenue.csv", index=False)

### 6️⃣ DELIVERY & SATISFACTION DATA

In [21]:
delivery_data = df[['delivery_time_days', 'review_score', 'total_order_value']].dropna()

delivery_data.to_csv("../data/processed/delivery_analysis.csv", index=False)